# DSEG-LIME adapté à la baseline Capuozzo 2D

Ce notebook constitue le **fil de démonstration**. Le code scientifique complet reste regroupé dans `dseg_lime_mri_probe.py`, afin d’éviter de dupliquer ou de disperser la logique.

**Principe :** SAM segmente la coupe FLAIR correspondante. DSEG-LIME construit des perturbations régionales, puis les mêmes masques sont appliqués à la coupe KLF réellement reçue par le CNN de Capuozzo.


## Chaîne expérimentale

`Volume KLF → coupe 2D → prédiction CNN → segmentation FLAIR par SAM → perturbations transférées à KLF → poids DSEG-LIME → comparaison future avec KBF`

La séparation FLAIR/KLF est volontaire : **FLAIR sert à construire des régions anatomiques**, tandis que **KLF reste l’entrée du classifieur**.


In [ ]:
# Cette cellule localise le dépôt et le script scientifique principal.
# Le notebook peut être ouvert depuis la racine du dépôt ou depuis contribution/.
from pathlib import Path

repo_dir = Path.cwd()
if repo_dir.name == "contribution":
    repo_dir = repo_dir.parent

script_path = repo_dir / "contribution" / "dseg_lime_mri_probe.py"
assert script_path.exists(), f"Script introuvable : {script_path}"

print("Dépôt :", repo_dir)
print("Script exécuté :", script_path)


## Exécuter l’explication complète

La cellule suivante est la seule cellule coûteuse. Elle charge le checkpoint, prépare KLF et FLAIR, exécute SAM, lance les perturbations DSEG-LIME et sauvegarde les résultats. Les commentaires précisent le rôle de chaque paramètre.


In [ ]:
# Paramètres de l'expérience.
# fold : partition de validation utilisée par Capuozzo.
# num_samples : nombre de perturbations DSEG-LIME par niveau.
# points_per_side : densité des points proposés à SAM.
# iterations : nombre de niveaux de raffinement DSEG.
fold = 0
num_samples = 128
points_per_side = 24
iterations = 2

# %run exécute le script complet dans Jupyter et conserve ses affichages ici.
%run "{script_path}" --fold {fold} --num-samples {num_samples} --points-per-side {points_per_side} --iterations {iterations}


### Ce que l’exécution vient de tester

- Le CNN reçoit toujours la modalité KLF attendue par la baseline.
- SAM reçoit la coupe FLAIR correspondante pour obtenir une segmentation exploitable.
- Chaque perturbation générée sur FLAIR est transformée en masque spatial puis transférée sur KLF.
- La variation de la probabilité du CNN permet à DSEG-LIME d’estimer l’importance de chaque région.

Les sorties de cette cellule prouvent que la chaîne logicielle fonctionne. Elles ne suffisent pas encore pour conclure à la validité clinique ou statistique de l’approche.


## Résultats numériques et affichages expliqués


In [ ]:
# Cette cellule regroupe les résultats numériques et les quatre sorties visuelles.
# Elle ne relance ni SAM ni DSEG-LIME : elle relit uniquement les fichiers sauvegardés.
import json
import matplotlib.pyplot as plt
from PIL import Image

output_dir = repo_dir / "contribution" / "outputs" / "patient_00003_fold0"
result = json.loads((output_dir / "result.json").read_text(encoding="utf-8"))

# Résumé interprétable du test.
p0, p1 = result["class_probabilities"]
print(f"Patient                  : {result['patient']}")
print(f"Fold                     : {result['fold']}")
print(f"Vérité terrain           : classe {result['label']}")
print(f"Coupe KLF sélectionnée   : {result['slice_index']}")
print(f"Voxels KLF non nuls      : {result['slice_nonzero_voxels']}")
print(f"P(classe 0)              : {p0:.4f}")
print(f"P(classe 1 / MGMT+)      : {p1:.4f}")
print(f"Classe expliquée         : {result['explained_class']}")
print(f"Nombre de régions DSEG   : {result['segments']}")
print()
print("Conclusion : prédiction correcte pour ce patient (vérité = 0, prédiction = 0).")
print("Attention  : un seul patient valide la chaîne technique, pas la méthode scientifique.")

# Les quatre images restent dans la même cellule que leur affichage.
files = ["klf_slice.png", "segments.png", "positive_mask.png", "weight_map.png"]
titles = [
    "1. Coupe utilisée",
    "2. Régions DSEG",
    "3. Régions positives",
    "4. Importance pondérée",
]
explanations = [
    "Entrée spatiale correspondant à la coupe 104",
    "Découpage produit par SAM puis raffiné par DSEG",
    "Régions favorisant la classe expliquée",
    "Poids local attribué à chaque région",
]

fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
for ax, filename, title, explanation in zip(axes, files, titles, explanations):
    ax.imshow(Image.open(output_dir / filename))
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(explanation, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(
    "Patient 00003 — de l'entrée KLF à l'explication DSEG-LIME",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()


### Interprétation du premier cas

Le patient `00003` a une vérité terrain égale à `0`. La probabilité de la classe 0 est `0,6465`, contre `0,3535` pour la classe MGMT positive : la prédiction est donc correcte sur ce cas.

DSEG-LIME produit seulement **3 régions finales**. Ce résultat est utile pour vérifier l’intégration, mais la segmentation est encore trop grossière pour soutenir une contribution scientifique. L’étape suivante consiste à régler la granularité, exécuter plusieurs patients et mesurer quantitativement l’accord entre DSEG-LIME, KBF et les masques tumoraux disponibles.


## Où lire le code détaillé ?

Le fichier `dseg_lime_mri_probe.py` contient toute la chaîne dans l’ordre d’exécution. Ses commentaires distinguent les six blocs : données, correspondance FLAIR/KLF, CNN, transfert des perturbations, SAM/DSEG-LIME et sauvegarde des résultats.

Cette organisation garde **le calcul dans un seul script** et **la démonstration avec ses affichages dans un seul notebook**.


## Étude pilote complète — 20 patients

Cette partie ne relance pas les calculs coûteux. Elle lit les résultats réellement produits pour **20 patients** équilibrés par classe et répartis sur les cinq folds. Elle rassemble la classification, la fidélité par suppression, les contrôles aléatoires, les courbes progressives, la stabilité et la sensibilité au nombre de perturbations.


In [ ]:
# Lecture et affichage groupés de tous les résultats du pilote à 20 patients.
import json
import pandas as pd
from IPython.display import display
from PIL import Image

pilot_dir = repo_dir / "contribution" / "pilot"
baseline = pd.read_csv(pilot_dir / "baseline_results.csv", dtype={"patient": str})
faithfulness = pd.read_csv(pilot_dir / "faithfulness_cases.csv", dtype={"patient": str})
complete = pd.read_csv(pilot_dir / "complete_evaluation_cases.csv", dtype={"patient": str})
stability = pd.read_csv(pilot_dir / "stability_summary.csv", dtype={"patient": str})
sensitivity = pd.read_csv(pilot_dir / "sample_sensitivity_summary.csv")
summary = json.loads((pilot_dir / "complete_evaluation_summary.json").read_text(encoding="utf-8"))

print("=== CLASSIFICATION SUR L'ÉCHANTILLON PILOTE ===")
print(f"Prédictions correctes : {int(baseline['correct'].sum())}/20 = {baseline['correct'].mean():.0%}")
print("Attention : ce petit échantillon teste l'explication ; il n'est pas une estimation finale de performance.\n")

print("=== FIDÉLITÉ DSEG-LIME ===")
print(f"Région DSEG principale = effet maximal : {faithfulness['top_region_is_most_faithful'].mean():.0%}")
print(f"Baisse positive après suppression : {(faithfulness['top_dseg_deletion_drop'] > 0).mean():.0%}")
print(f"Spearman médian poids/effet : {faithfulness['weight_drop_spearman'].median():.3f}\n")

print("=== CONTRÔLES ET COURBES ===")
print(f"DSEG > suppression aléatoire moyenne : {summary['top_beats_random_mean_rate']:.0%}")
print(f"Contrôle empirique p<=0,05 : {summary['top_significant_vs_random_rate_p05']:.0%}")
print(f"Suppression progressive meilleure : {summary['deletion_better_than_random_rate']:.0%}")
print(f"Insertion progressive meilleure : {summary['insertion_better_than_random_rate']:.0%}\n")

print("=== STABILITÉ SUR 6 PATIENTS, 3 GRAINES ===")
display(stability.round(3))
print("=== SENSIBILITÉ 64 / 128 / 256 ===")
display(sensitivity.round(3))
print("Conclusion : continuer DSEG-LIME, mais utiliser au moins 128 perturbations pour les cas à nombreuses régions.")

display(Image.open(pilot_dir / "pilot_complete_summary.png"))


### Conclusion opérationnelle du pilote

Les 20 patients apportent un signal suffisamment cohérent pour **continuer l’approche** : la région principale est fidèle, elle dépasse généralement un contrôle aléatoire de même taille utile, et les ordres de régions DSEG améliorent les courbes de suppression/insertion.

La comparaison spatiale brute avec KBF doit être interprétée avec prudence : le rappel médian de KBF par DSEG positif est élevé (`0,865`), mais DSEG couvre des régions FLAIR bien plus grandes que le masque KBF parcimonieux. La contribution la plus défendable est donc un **KBF repondéré à l’intérieur de son propre support**, guidé par les poids DSEG, puis évalué contre le KBF original sans changer le CNN ni les folds.
